# ARC-2 grid tokenizer + training corpus — run this first

Emits the input dataset for a training run whose output plugs into **Qwen3.5-35B-A3B V1** as a Phase A model.

## The codec

Three lossless schemes over one string format. Sequence length is what sets both TTT cost and DFS cost, so it is the axis worth optimising — and because every ARC grid is at most 30×30 over ten colours, the whole content space is known in advance: the codec can be dense *and* exact, with no learned merges and nothing out-of-distribution.

Measured on the competition's 1,000 training tasks (300 sampled, tokens per full task):

| scheme | vocab | p50 | p90 | max | vs g16 | compatible with the sorokin 4B |
|---|---|---|---|---|---|---|
| `g16` | 16 | 1100 | 3341 | 7102 | 1.00× | **yes — drop-in** |
| `g116` | 116 | 641 | 1818 | 3744 | **1.72×** | no, needs a model trained on it |
| `g1116` | 1016 | 520 | 1375 | 2816 | **2.12×** | no, needs a model trained on it |

`k` cells are packed per token, **never across a row boundary** — a row of width W emits W//k grams plus its leftover cells as singles, then the separator. A flat BPE over the whole grid would compress harder and destroy the row structure that makes the 2D task learnable at all. Round-trip against the reference `g16` string is asserted for all three schemes on 300 real tasks before anything is written; if it ever fails the notebook stops instead of shipping a corrupt corpus. An optional `<\|shape\|>H\nW<\|shape\|>` header (`ARC_TOK_SHAPE_HEADER=1`) states each grid's dimensions up front.

## Honest limits

`g116` and `g1116` are **not usable by your current Phase A model** — its embedding table is bound to the 16 ids, so a wider vocab means training from the SFT stage, not swapping a file in. They are here because they are the format a *new* model should be trained on: 1.7–2.1× shorter sequences means proportionally cheaper TTT and DFS, and an hour of Phase A is worth roughly 1.25 leaderboard points. There is no vision tokenizer here: Phase A's model is a text LM with no image encoder, and patch tokens need a different architecture and a full pretrain, which four L4s cannot do.

## Outputs

`/kaggle/working/arc_grid_tok/` — `tokenizer/grid_tokenizer.json` (vocab, ids, compression table, a `compat` field the main notebook reads), `tokenizer/arc_grid_codec.py` (the exact encode/decode used to build the corpus, so training and inference cannot drift), `train.jsonl` / `valid.jsonl`, `corpus_stats.json`.

## Sources and leakage

Every ARC-format json under `/kaggle/input` is scanned and deduplicated by content, with test outputs spliced back in from any sibling `*_solutions.json`; a task with no answer key holds out its last demonstration instead. Attach whatever you have — ARC-AGI-1 train/eval, ARC-AGI-2 training, re-arc, ConceptARC, `sorokin/nvarc-synthetic-puzzles`. **This competition's evaluation and test splits are excluded by name.** Training on them prints a fake local score, collapses on the hidden set, and is the first thing a prize review checks.

## Settings

Accelerator **None** (CPU is enough). Internet off. `ARC_TOK_SCHEME` `g16` · `ARC_TOK_AUG` 8 · `ARC_TOK_MAX_SEQ` 8192 · `ARC_TOK_MAX_SAMPLES` 400000 · `ARC_TOK_SHAPE_HEADER` 0 · `ARC_TOK_SEED` 0.

When it finishes: **Save Version → Output → New Dataset**, then attach that dataset to the training notebook. The main notebook also detects the artifact and prints whether its vocab matches the attached grid model.

In [ ]:
import os, sys, json, glob, hashlib, random, collections
from pathlib import Path

OUT = "/kaggle/working/arc_grid_tok"
os.makedirs(OUT, exist_ok=True)
SEED = int(os.getenv("ARC_TOK_SEED", "0"))
random.seed(SEED)

# ── the format the sorokin 4B was trained on (arc_loader.QwenFormatter / convert_grid_to_string) ──
#   grid  -> one digit per cell, rows separated by "\n", trailing "\n"
#   pair  -> <|im_start|>user\n{input}<|im_end|><|im_start|>assistant\n{output}<|im_end|>
#   vocab -> 16 ids: 10 digits, "\n", pad, and the three chat control tokens
GRID_VOCAB = {str(d): d for d in range(10)}
GRID_VOCAB.update({"\n": 10, "<|pad|>": 11, "<|im_start|>": 12, "user": 13, "assistant": 14, "<|im_end|>": 15})


def grid_to_string(g):
    return "".join("".join(str(int(c)) for c in row) + "\n" for row in g)


def fmt_pair(inp, out=None):
    s = "<|im_start|>user\n" + grid_to_string(inp) + "<|im_end|><|im_start|>assistant\n"
    return s + (grid_to_string(out) + "<|im_end|>" if out is not None else "")


def fmt_task(train_pairs, query, answer=None):
    return "".join(fmt_pair(p["input"], p["output"]) for p in train_pairs) + fmt_pair(query, answer)


def valid_grid(g):
    return (isinstance(g, list) and 1 <= len(g) <= 30 and all(isinstance(r, list) for r in g)
            and 1 <= len(g[0]) <= 30 and all(len(r) == len(g[0]) for r in g)
            and all(isinstance(c, int) and 0 <= c <= 9 for r in g for c in r))


def normalise_task(t, sol=None):
    """Return {'train': [...], 'test': [one pair with an output]} or None.

    ARC *challenge* files carry test inputs only - the outputs live in the sibling *_solutions.json. When the
    solutions file is present we splice them back in; when it is not, the task is still usable by treating its
    last training pair as the query, which is exactly what the solver does during test-time training.
    """
    if not isinstance(t, dict) or "train" not in t:
        return None
    train = [p for p in t.get("train", []) if isinstance(p, dict)
             and valid_grid(p.get("input")) and valid_grid(p.get("output"))]
    tests = []
    for i, p in enumerate(t.get("test", []) or []):
        if not isinstance(p, dict) or not valid_grid(p.get("input")):
            continue
        out = p.get("output")
        if out is None and sol is not None and i < len(sol):
            out = sol[i]
        if valid_grid(out):
            tests.append({"input": p["input"], "output": out})
    if not tests and len(train) >= 3:
        tests, train = [train[-1]], train[:-1]        # no answer key: hold out the last demonstration
    if len(train) < 2 or len(train) > 10 or not tests:
        return None
    return {"train": train, "test": tests[:1]}


# ── geometry x colour augmentation, the same space the solver samples at test time ──
def rot90(g):
    return [list(r) for r in zip(*g[::-1])]


def transpose(g):
    return [list(r) for r in zip(*g)]


def apply_geom(g, k, t):
    for _ in range(k):
        g = rot90(g)
    return transpose(g) if t else g


def apply_perm(g, perm):
    return [[perm[c] for c in row] for row in g]


def augment_task(t, rng, keep_zero=True):
    k, tr = rng.randrange(4), rng.random() < 0.5
    colors = list(range(1, 10))
    rng.shuffle(colors)
    perm = [0] + colors if keep_zero else rng.sample(range(10), 10)
    ex = [{"input": apply_perm(apply_geom(p["input"], k, tr), perm),
           "output": apply_perm(apply_geom(p["output"], k, tr), perm)} for p in t["train"] + t["test"]]
    rng.shuffle(ex[:-1])
    return {"train": ex[:-1], "test": [ex[-1]]}


# ── sources: every ARC-format json under /kaggle/input, deduplicated by content ──
def load_sources():
    """Every ARC-format json under /kaggle/input, deduplicated by content.

    The evaluation and test splits of this competition are excluded by name. Training on them would print a
    fake local score and collapse on the hidden set, and it is the first thing a prize review checks.
    """
    seen, tasks, per_source = set(), [], collections.Counter()
    comp = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2"
    banned = {f"{comp}/arc-agi_evaluation_challenges.json", f"{comp}/arc-agi_evaluation_solutions.json",
              f"{comp}/arc-agi_test_challenges.json"}
    paths = [p for p in glob.glob("/kaggle/input/**/*.json", recursive=True)
             if "sample_submission" not in p and "solutions" not in os.path.basename(p)]
    for p in sorted(paths):
        if p in banned or os.path.getsize(p) > 800 * 1024 * 1024:
            continue
        try:
            data = json.load(open(p))
        except Exception:
            continue
        sols = {}
        sib = p.replace("challenges", "solutions")
        if sib != p and sib not in banned and os.path.exists(sib):
            try:
                sols = json.load(open(sib))
            except Exception:
                sols = {}
        items = data.items() if isinstance(data, dict) else enumerate(data)
        n = 0
        for key, t in items:
            nt = normalise_task(t, sols.get(key) if isinstance(sols, dict) else None)
            if nt is None:
                continue
            h = hashlib.md5(json.dumps(nt, sort_keys=True).encode()).hexdigest()
            if h in seen:
                continue
            seen.add(h); tasks.append(nt); n += 1
        if n:
            per_source[p] = n
    return tasks, per_source


print("scanning /kaggle/input for ARC-format task files ...")
tasks, per_source = load_sources()
for p, n in per_source.most_common(20):
    print(f"  {n:7d}  {p}")
print(f"total unique tasks: {len(tasks)}")


In [ ]:
"""Grid codec. Three lossless schemes over the same string format; pick one with ARC_TOK_SCHEME.

    g16     1 token per cell            vocab   16   the format the sorokin 4B was trained on (default, drop-in)
    g116    2 cells per token           vocab  116   ~1.9x shorter
    g1116   3 cells per token           vocab 1116   ~2.8x shorter

Packing never crosses a row boundary: a row of width W emits W//k grams plus the 1-2 leftover cells as single
tokens, then the row separator. Row structure stays visible to the model, which is the property that makes the
2D task learnable at all - a flat BPE over the whole grid would compress harder and destroy it.

Why this is the axis worth optimising: sequence length sets both TTT cost and DFS cost. Every ARC grid is at
most 30x30, so the entire content of a task is known in advance and the codec can be dense *and* exact; there
is no vocabulary to learn and nothing to be out-of-distribution. Measured below on the real corpus.
"""
import numpy as np

SCHEME = os.getenv("ARC_TOK_SCHEME", "g16")
SHAPE_HEADER = os.getenv("ARC_TOK_SHAPE_HEADER", "0") == "1"
MAX_SEQ = int(os.getenv("ARC_TOK_MAX_SEQ", "8192"))
AUG_PER_TASK = int(os.getenv("ARC_TOK_AUG", "8"))
MAX_SAMPLES = int(os.getenv("ARC_TOK_MAX_SAMPLES", "400000"))
K_OF = {"g16": 1, "g116": 2, "g1116": 3}
assert SCHEME in K_OF, f"ARC_TOK_SCHEME must be one of {sorted(K_OF)}"
K = K_OF[SCHEME]

CTRL = ["<|pad|>", "<|im_start|>", "user", "assistant", "<|im_end|>", "<|shape|>"]


def build_vocab(k, shape_header):
    """id layout: 0-9 single cells, 10 row separator, 11.. control, then the k-gram block."""
    v = {str(d): d for d in range(10)}
    v["\n"] = 10
    nxt = 11
    for c in CTRL:
        if c == "<|shape|>" and not shape_header:
            continue
        v[c] = nxt; nxt += 1
    grams = {}
    if k > 1:
        for i in range(10 ** k):
            g = str(i).zfill(k)
            v[g] = nxt; grams[g] = nxt; nxt += 1
    return v, grams, nxt


GRID_VOCAB, GRAMS, VOCAB_SIZE = build_vocab(K, SHAPE_HEADER)
INV_VOCAB = {i: s for s, i in GRID_VOCAB.items()}
EOS_ID, PAD_ID = GRID_VOCAB["<|im_end|>"], GRID_VOCAB["<|pad|>"]


def encode_grid(g):
    ids = []
    if SHAPE_HEADER:
        ids.append(GRID_VOCAB["<|shape|>"])
        ids += [GRID_VOCAB[c] for c in f"{len(g)}"] + [GRID_VOCAB["\n"]] + [GRID_VOCAB[c] for c in f"{len(g[0])}"]
        ids.append(GRID_VOCAB["<|shape|>"])
    for row in g:
        s = "".join(str(int(c)) for c in row)
        i = 0
        while i + K <= len(s):
            ids.append(GRID_VOCAB[s[i:i + K]] if K > 1 else GRID_VOCAB[s[i]])
            i += K
        while i < len(s):                      # 1-2 leftover cells at the end of a row
            ids.append(GRID_VOCAB[s[i]]); i += 1
        ids.append(GRID_VOCAB["\n"])
    return ids


def decode_grid_ids(ids):
    """Inverse of encode_grid; returns the digit string with row separators, ready to split."""
    out, i = [], 0
    if SHAPE_HEADER and ids and ids[0] == GRID_VOCAB["<|shape|>"]:
        i = ids.index(GRID_VOCAB["<|shape|>"], 1) + 1
    for t in ids[i:]:
        out.append(INV_VOCAB[t])
    return "".join(out)


def encode(pairs):
    """pairs: list of (role, grid) with role in {'user','assistant'}; grid None closes an open assistant turn."""
    ids = []
    for role, grid in pairs:
        ids += [GRID_VOCAB["<|im_start|>"], GRID_VOCAB[role], GRID_VOCAB["\n"]]
        if grid is None:
            continue
        ids += encode_grid(grid) + [GRID_VOCAB["<|im_end|>"]]
    return ids


def task_to_pairs(train_pairs, query, answer=None):
    seq = []
    for p in train_pairs:
        seq += [("user", p["input"]), ("assistant", p["output"])]
    seq += [("user", query), ("assistant", answer)]
    return seq


def decode(ids):
    """Full round-trip: token ids -> the same string a g16 formatter would have produced."""
    parts, i = [], 0
    while i < len(ids):
        t = ids[i]
        if t == GRID_VOCAB["<|im_start|>"]:
            role = INV_VOCAB[ids[i + 1]]
            parts.append(f"<|im_start|>{role}\n"); i += 3
            body = []
            while i < len(ids) and ids[i] != GRID_VOCAB["<|im_end|>"] and ids[i] != GRID_VOCAB["<|im_start|>"]:
                body.append(ids[i]); i += 1
            parts.append(decode_grid_ids(body))
            if i < len(ids) and ids[i] == GRID_VOCAB["<|im_end|>"]:
                parts.append("<|im_end|>"); i += 1
        else:
            parts.append(INV_VOCAB[t]); i += 1
    return "".join(parts)


def to_text(train_pairs, query, answer=None):
    """The g16 reference string - what a model with the sorokin vocab consumes verbatim."""
    def gs(g):
        return "".join("".join(str(int(c)) for c in row) + "\n" for row in g)
    s = "".join(f"<|im_start|>user\n{gs(p['input'])}<|im_end|><|im_start|>assistant\n{gs(p['output'])}<|im_end|>"
                for p in train_pairs)
    s += f"<|im_start|>user\n{gs(query)}<|im_end|><|im_start|>assistant\n"
    return s + (gs(answer) + "<|im_end|>" if answer is not None else "")


In [ ]:
# The codec above, captured verbatim so the exported artifact contains the exact encode/decode used here.
CODEC_SRC = '"""Grid codec. Three lossless schemes over the same string format; pick one with ARC_TOK_SCHEME.\n\n    g16     1 token per cell            vocab   16   the format the sorokin 4B was trained on (default, drop-in)\n    g116    2 cells per token           vocab  116   ~1.9x shorter\n    g1116   3 cells per token           vocab 1116   ~2.8x shorter\n\nPacking never crosses a row boundary: a row of width W emits W//k grams plus the 1-2 leftover cells as single\ntokens, then the row separator. Row structure stays visible to the model, which is the property that makes the\n2D task learnable at all - a flat BPE over the whole grid would compress harder and destroy it.\n\nWhy this is the axis worth optimising: sequence length sets both TTT cost and DFS cost. Every ARC grid is at\nmost 30x30, so the entire content of a task is known in advance and the codec can be dense *and* exact; there\nis no vocabulary to learn and nothing to be out-of-distribution. Measured below on the real corpus.\n"""\nimport numpy as np\n\nSCHEME = os.getenv("ARC_TOK_SCHEME", "g16")\nSHAPE_HEADER = os.getenv("ARC_TOK_SHAPE_HEADER", "0") == "1"\nMAX_SEQ = int(os.getenv("ARC_TOK_MAX_SEQ", "8192"))\nAUG_PER_TASK = int(os.getenv("ARC_TOK_AUG", "8"))\nMAX_SAMPLES = int(os.getenv("ARC_TOK_MAX_SAMPLES", "400000"))\nK_OF = {"g16": 1, "g116": 2, "g1116": 3}\nassert SCHEME in K_OF, f"ARC_TOK_SCHEME must be one of {sorted(K_OF)}"\nK = K_OF[SCHEME]\n\nCTRL = ["<|pad|>", "<|im_start|>", "user", "assistant", "<|im_end|>", "<|shape|>"]\n\n\ndef build_vocab(k, shape_header):\n    """id layout: 0-9 single cells, 10 row separator, 11.. control, then the k-gram block."""\n    v = {str(d): d for d in range(10)}\n    v["\\n"] = 10\n    nxt = 11\n    for c in CTRL:\n        if c == "<|shape|>" and not shape_header:\n            continue\n        v[c] = nxt; nxt += 1\n    grams = {}\n    if k > 1:\n        for i in range(10 ** k):\n            g = str(i).zfill(k)\n            v[g] = nxt; grams[g] = nxt; nxt += 1\n    return v, grams, nxt\n\n\nGRID_VOCAB, GRAMS, VOCAB_SIZE = build_vocab(K, SHAPE_HEADER)\nINV_VOCAB = {i: s for s, i in GRID_VOCAB.items()}\nEOS_ID, PAD_ID = GRID_VOCAB["<|im_end|>"], GRID_VOCAB["<|pad|>"]\n\n\ndef encode_grid(g):\n    ids = []\n    if SHAPE_HEADER:\n        ids.append(GRID_VOCAB["<|shape|>"])\n        ids += [GRID_VOCAB[c] for c in f"{len(g)}"] + [GRID_VOCAB["\\n"]] + [GRID_VOCAB[c] for c in f"{len(g[0])}"]\n        ids.append(GRID_VOCAB["<|shape|>"])\n    for row in g:\n        s = "".join(str(int(c)) for c in row)\n        i = 0\n        while i + K <= len(s):\n            ids.append(GRID_VOCAB[s[i:i + K]] if K > 1 else GRID_VOCAB[s[i]])\n            i += K\n        while i < len(s):                      # 1-2 leftover cells at the end of a row\n            ids.append(GRID_VOCAB[s[i]]); i += 1\n        ids.append(GRID_VOCAB["\\n"])\n    return ids\n\n\ndef decode_grid_ids(ids):\n    """Inverse of encode_grid; returns the digit string with row separators, ready to split."""\n    out, i = [], 0\n    if SHAPE_HEADER and ids and ids[0] == GRID_VOCAB["<|shape|>"]:\n        i = ids.index(GRID_VOCAB["<|shape|>"], 1) + 1\n    for t in ids[i:]:\n        out.append(INV_VOCAB[t])\n    return "".join(out)\n\n\ndef encode(pairs):\n    """pairs: list of (role, grid) with role in {\'user\',\'assistant\'}; grid None closes an open assistant turn."""\n    ids = []\n    for role, grid in pairs:\n        ids += [GRID_VOCAB["<|im_start|>"], GRID_VOCAB[role], GRID_VOCAB["\\n"]]\n        if grid is None:\n            continue\n        ids += encode_grid(grid) + [GRID_VOCAB["<|im_end|>"]]\n    return ids\n\n\ndef task_to_pairs(train_pairs, query, answer=None):\n    seq = []\n    for p in train_pairs:\n        seq += [("user", p["input"]), ("assistant", p["output"])]\n    seq += [("user", query), ("assistant", answer)]\n    return seq\n\n\ndef decode(ids):\n    """Full round-trip: token ids -> the same string a g16 formatter would have produced."""\n    parts, i = [], 0\n    while i < len(ids):\n        t = ids[i]\n        if t == GRID_VOCAB["<|im_start|>"]:\n            role = INV_VOCAB[ids[i + 1]]\n            parts.append(f"<|im_start|>{role}\\n"); i += 3\n            body = []\n            while i < len(ids) and ids[i] != GRID_VOCAB["<|im_end|>"] and ids[i] != GRID_VOCAB["<|im_start|>"]:\n                body.append(ids[i]); i += 1\n            parts.append(decode_grid_ids(body))\n            if i < len(ids) and ids[i] == GRID_VOCAB["<|im_end|>"]:\n                parts.append("<|im_end|>"); i += 1\n        else:\n            parts.append(INV_VOCAB[t]); i += 1\n    return "".join(parts)\n\n\ndef to_text(train_pairs, query, answer=None):\n    """The g16 reference string - what a model with the sorokin vocab consumes verbatim."""\n    def gs(g):\n        return "".join("".join(str(int(c)) for c in row) + "\\n" for row in g)\n    s = "".join(f"<|im_start|>user\\n{gs(p[\'input\'])}<|im_end|><|im_start|>assistant\\n{gs(p[\'output\'])}<|im_end|>"\n                for p in train_pairs)\n    s += f"<|im_start|>user\\n{gs(query)}<|im_end|><|im_start|>assistant\\n"\n    return s + (gs(answer) + "<|im_end|>" if answer is not None else "")\n'


In [ ]:
rng = random.Random(SEED)
probe = rng.sample(tasks, min(300, len(tasks)))

# ── correctness first: every scheme must reproduce the g16 reference string exactly ──
for name, k in sorted(K_OF.items()):
    V, G, VS = build_vocab(k, SHAPE_HEADER)
    saved = (GRID_VOCAB, GRAMS, VOCAB_SIZE, INV_VOCAB, K)
    globals().update(GRID_VOCAB=V, GRAMS=G, VOCAB_SIZE=VS, INV_VOCAB={i: s for s, i in V.items()}, K=k)
    bad = 0
    for t in probe:
        ref = to_text(t["train"], t["test"][0]["input"], t["test"][0]["output"])
        if decode(encode(task_to_pairs(t["train"], t["test"][0]["input"], t["test"][0]["output"]))) != ref:
            bad += 1
    assert bad == 0, f"{name}: {bad}/{len(probe)} round-trip failures"
    globals().update(GRID_VOCAB=saved[0], GRAMS=saved[1], VOCAB_SIZE=saved[2], INV_VOCAB=saved[3], K=saved[4])
print(f"lossless round-trip verified for all schemes on {len(probe)} real tasks")

# ── compression table on the real corpus ──
print(f"\n{'scheme':8s} {'vocab':>6s} {'p50':>7s} {'p90':>7s} {'max':>7s} {'vs g16':>7s} {'over 8192':>10s}")
table = {}
base = None
for name, k in sorted(K_OF.items(), key=lambda kv: kv[1]):
    V, G, VS = build_vocab(k, SHAPE_HEADER)
    saved = (GRID_VOCAB, GRAMS, VOCAB_SIZE, INV_VOCAB, K)
    globals().update(GRID_VOCAB=V, GRAMS=G, VOCAB_SIZE=VS, INV_VOCAB={i: s for s, i in V.items()}, K=k)
    L = [len(encode(task_to_pairs(t["train"], t["test"][0]["input"], t["test"][0]["output"]))) for t in probe]
    globals().update(GRID_VOCAB=saved[0], GRAMS=saved[1], VOCAB_SIZE=saved[2], INV_VOCAB=saved[3], K=saved[4])
    p50, p90, mx = (int(np.percentile(L, 50)), int(np.percentile(L, 90)), max(L))
    base = base or p50
    table[name] = {"vocab": VS, "p50": p50, "p90": p90, "max": mx, "ratio": round(base / p50, 2),
                   "over_max_seq": sum(x > MAX_SEQ for x in L)}
    print(f"{name:8s} {VS:6d} {p50:7d} {p90:7d} {mx:7d} {base/p50:6.2f}x {table[name]['over_max_seq']:10d}")

# ── artifacts: vocab, a reference codec the training run imports verbatim, and the corpus ──
tok_dir = Path(OUT, "tokenizer"); tok_dir.mkdir(parents=True, exist_ok=True)
(tok_dir / "arc_grid_codec.py").write_text(CODEC_SRC)
json.dump({"scheme": SCHEME, "k": K, "shape_header": SHAPE_HEADER, "vocab": GRID_VOCAB, "vocab_size": VOCAB_SIZE,
           "eos_id": EOS_ID, "pad_id": PAD_ID, "max_seq": MAX_SEQ,
           "reference_format": "<|im_start|>user\\n{grid}<|im_end|><|im_start|>assistant\\n{grid}<|im_end|>",
           "grid": "one digit per cell, rows joined by \\n with a trailing \\n; k cells packed per token, "
                   "never across a row boundary",
           "compression": table,
           "compat": ("drop-in for the sorokin 4B (vocab 16)" if SCHEME == "g16" else
                      "requires a model trained on this vocab - NOT loadable by the existing Phase A 4B"),
           "codec": "arc_grid_codec.py in this folder is the exact encode/decode used to build the corpus"},
          open(tok_dir / "grid_tokenizer.json", "w"), indent=1)

rows, skipped = [], 0
order = sorted(range(len(tasks)), key=lambda i: -sum(len(p["input"]) for p in tasks[i]["train"]))
for idx in order:
    t = tasks[idx]
    for a in range(AUG_PER_TASK):
        at = augment_task(t, rng) if a else {"train": t["train"], "test": [t["test"][0]]}
        q, ans = at["test"][0]["input"], at["test"][0]["output"]
        ids = encode(task_to_pairs(at["train"], q, ans))
        if len(ids) > MAX_SEQ:
            skipped += 1; continue
        rows.append({"text": to_text(at["train"], q, ans), "ids": ids} if SCHEME != "g16"
                    else {"text": to_text(at["train"], q, ans)})
        if len(rows) >= MAX_SAMPLES:
            break
    if len(rows) >= MAX_SAMPLES:
        break
rng.shuffle(rows)
n_val = min(2000, len(rows) // 20)
for name, part in {"valid": rows[:n_val], "train": rows[n_val:]}.items():
    with open(Path(OUT, f"{name}.jsonl"), "w") as f:
        for r in part:
            f.write(json.dumps(r) + "\n")
stats = {"scheme": SCHEME, "tasks": len(tasks), "samples": len(rows), "valid": n_val,
         "skipped_too_long": skipped, "aug_per_task": AUG_PER_TASK, "max_seq": MAX_SEQ, "seed": SEED,
         "compression": table, "sources": {os.path.basename(p): n for p, n in per_source.most_common()}}
json.dump(stats, open(Path(OUT, "corpus_stats.json"), "w"), indent=1)
print(f"\nscheme {SCHEME} | {len(rows)} samples from {len(tasks)} tasks | {skipped} over {MAX_SEQ} tokens")
print(f"written to {OUT}: tokenizer/grid_tokenizer.json, tokenizer/arc_grid_codec.py, train.jsonl, valid.jsonl")
print("Save Version -> Output -> 'New Dataset', then attach that dataset to the training notebook.")
